In [0]:
# ============================================================
# SILVER LAYER: Clean, typed, deduplicated watch history
# ============================================================
# Read from bronze, apply cleaning rules, write to silver.
# Goal: every column typed correctly, every row trustworthy.
# ============================================================

from pyspark.sql.functions import (
    col, regexp_extract, to_timestamp, when, lit,
    year, month, dayofweek, hour, date_format,
    lower, trim, length, current_timestamp,
    row_number
)
from pyspark.sql.window import Window

BRONZE_TABLE = "youtube_wrapped.bronze.watch_history_raw"
SILVER_TABLE = "youtube_wrapped.silver.watch_history_clean"

print(f"Source: {BRONZE_TABLE}")
print(f"Target: {SILVER_TABLE}")

In [0]:
df_bronze = spark.table(BRONZE_TABLE)

print(f"Bronze row count: {df_bronze.count():,}")
print("\n--- Rows WITHOUT a titleUrl (likely deleted/private videos or ads) ---")
df_bronze.filter(col("titleUrl").isNull()).show(5, truncate=100)

print("\n--- Rows WITHOUT subtitles (no channel info) ---")
df_bronze.filter(col("subtitles").isNull()).show(5, truncate=100)

In [0]:
# Extract channel name and URL from the nested subtitles array
# subtitles is an array, but it always has exactly 0 or 1 element in practice
df_with_channel = (
    df_bronze
    .withColumn("channel_name", col("subtitles").getItem(0).getField("name"))
    .withColumn("channel_url", col("subtitles").getItem(0).getField("url"))
)

# Extract video_id from titleUrl using regex
# URL format: https://www.youtube.com/watch?v=<11-char-id>
VIDEO_ID_REGEX = r"v=([a-zA-Z0-9_-]{11})"
# Extract channel_id from channel_url
# URL format: https://www.youtube.com/channel/<id>
CHANNEL_ID_REGEX = r"/channel/([a-zA-Z0-9_-]+)"

df_extracted = (
    df_with_channel
    .withColumn("video_id", regexp_extract(col("titleUrl"), VIDEO_ID_REGEX, 1))
    .withColumn("channel_id", regexp_extract(col("channel_url"), CHANNEL_ID_REGEX, 1))
)

# Clean the title: strip "Watched " prefix
# Takeout prefixes every watched video with "Watched <title>"
df_cleaned_title = (
    df_extracted
    .withColumn(
        "video_title",
        when(col("title").startswith("Watched "), 
             col("title").substr(lit(9), length(col("title")) - 8))
        .otherwise(col("title"))
    )
)

# Parse timestamp — Takeout gives ISO 8601 strings in UTC
df_typed = (
    df_cleaned_title
    .withColumn("watched_at_utc", to_timestamp(col("time")))
)

# Add time-derived columns — these power the fun-fact aggregations later
df_temporal = (
    df_typed
    .withColumn("watch_year", year(col("watched_at_utc")))
    .withColumn("watch_month", month(col("watched_at_utc")))
    .withColumn("watch_day_of_week", dayofweek(col("watched_at_utc")))  # 1=Sunday
    .withColumn("watch_hour", hour(col("watched_at_utc")))
    .withColumn("watch_day_name", date_format(col("watched_at_utc"), "EEEE"))
)

# Flag music: YouTube auto-generates "<Artist Name> - Topic" channels for music
# This is the cleanest music signal in Takeout data
df_flagged = (
    df_temporal
    .withColumn(
        "is_music_topic",
        when(col("channel_name").endswith(" - Topic"), lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "artist_name_from_topic",
        when(col("channel_name").endswith(" - Topic"),
             col("channel_name").substr(lit(1), length(col("channel_name")) - 8))
        .otherwise(lit(None))
    )
)

print("Transformations applied. Preview:")
df_flagged.select(
    "video_id", "video_title", "channel_name", 
    "watched_at_utc", "is_music_topic", "artist_name_from_topic"
).show(5, truncate=40)

In [0]:
# Drop rows that aren't real watches:
# - No video_id means deleted/private video or non-watch activity
# - No watched_at_utc means the timestamp failed to parse
# - "Watched a video that has been removed" is Takeout's placeholder
df_filtered = (
    df_flagged
    .filter(col("video_id") != "")                 # must have extracted a video_id
    .filter(col("watched_at_utc").isNotNull())     # must have valid timestamp
    .filter(~col("video_title").contains("a video that has been removed"))
)

rows_before = df_flagged.count()
rows_after = df_filtered.count()
rows_dropped = rows_before - rows_after

print(f"Before filter: {rows_before:,}")
print(f"After filter:  {rows_after:,}")
print(f"Dropped:       {rows_dropped:,} ({100 * rows_dropped / rows_before:.2f}%)")

In [0]:
# Dedupe: same video_id + same timestamp = same watch event
# Keep the earliest-ingested version
dedup_window = Window.partitionBy("video_id", "watched_at_utc").orderBy("_ingested_at")

df_deduped = (
    df_filtered
    .withColumn("_row_num", row_number().over(dedup_window))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

duplicates_removed = df_filtered.count() - df_deduped.count()
print(f"Exact duplicates removed: {duplicates_removed:,}")

In [0]:
df_silver = df_deduped.select(
    # Core identifiers
    "video_id",
    "video_title",
    "channel_id",
    "channel_name",
    
    # Temporal
    "watched_at_utc",
    "watch_year",
    "watch_month",
    "watch_day_of_week",
    "watch_day_name",
    "watch_hour",
    
    # URLs (useful for debugging and frontend links)
    col("titleUrl").alias("video_url"),
    "channel_url",
    
    # Music signal
    "is_music_topic",
    "artist_name_from_topic",
    
    # Lineage
    "_ingested_at",
).withColumn("_silver_processed_at", current_timestamp())

print("Final silver schema:")
df_silver.printSchema()

print("\nSample rows:")
df_silver.show(5, truncate=50)

In [0]:
(
    df_silver.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"✅ Wrote {df_silver.count():,} rows to {SILVER_TABLE}")

In [0]:
%sql
-- How many watches total?
SELECT COUNT(*) AS total_watches FROM youtube_wrapped.silver.watch_history_clean;

-- How many are music?
SELECT 
  is_music_topic,
  COUNT(*) AS watches,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM youtube_wrapped.silver.watch_history_clean
GROUP BY is_music_topic;

-- Top 10 channels by watch count
SELECT channel_name, COUNT(*) AS watches
FROM youtube_wrapped.silver.watch_history_clean
GROUP BY channel_name
ORDER BY watches DESC
LIMIT 10;

-- Top 10 artists (music only)
SELECT artist_name_from_topic AS artist, COUNT(*) AS plays
FROM youtube_wrapped.silver.watch_history_clean
WHERE is_music_topic = true
GROUP BY artist_name_from_topic
ORDER BY plays DESC
LIMIT 10;

-- When do you watch? (hour of day distribution)
SELECT watch_hour, COUNT(*) AS watches
FROM youtube_wrapped.silver.watch_history_clean
GROUP BY watch_hour
ORDER BY watch_hour;